# Netflix Content Strategy — EDA & Business Recommendations
**Data Analyst Portfolio Project**  
Hugo Apolinário · 2025

**Business question:** What types of content should Netflix invest in to maximise audience reach and satisfaction?

This notebook analyses 8,800+ Netflix titles to uncover patterns in content type, genre, country, release timing, and ratings — and translates those findings into concrete strategic recommendations.

---
## 0. Setup — install and import libraries
Run this cell first every time you open this notebook.

In [ ]:
import micropip
await micropip.install('seaborn')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

print('Libraries loaded successfully!')

---
## 1. Load the data
Upload `netflix_titles.csv` to JupyterLab first (drag it into the file browser on the left sidebar), then run this cell.

In [ ]:
# Load the CSV file into a DataFrame (think of it as an Excel sheet in Python)
df = pd.read_csv('netflix_titles.csv')

# Show the first 5 rows so we can see what the data looks like
print(f'Dataset loaded: {df.shape[0]} rows and {df.shape[1]} columns')
df.head()

---
## 2. Data cleaning
Real-world data is always messy. Here we fix the issues before analysing.

In [ ]:
# --- STEP 1: Check for missing values ---
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# --- STEP 2: Fix missing values ---

# Fill missing text columns with 'Unknown' so they don't break our analysis
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna('Unknown', inplace=True)

# Drop rows where duration is missing (very few, not worth keeping)
df.dropna(subset=['duration'], inplace=True)

# --- STEP 3: Fix the date column ---
# Convert 'date_added' to a proper date format Python understands
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

# Extract the year and month added as separate columns — useful for trend analysis
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

# --- STEP 4: Clean the duration column ---
# Movies are in minutes, TV shows are in seasons — we separate them
movies = df[df['type'] == 'Movie'].copy()
shows  = df[df['type'] == 'TV Show'].copy()

# Convert movie duration to a number (remove the word 'min')
movies['duration_mins'] = movies['duration'].str.replace(' min', '').astype(float)

# Convert TV show duration to number of seasons
shows['duration_seasons'] = shows['duration'].str.replace(' Season', '').str.replace('s', '').astype(float)

print('Data cleaning complete!')
print(f'Movies: {len(movies)} | TV Shows: {len(shows)}')

---
## 3. Exploratory analysis
### 3a. Movies vs TV shows — what does Netflix focus on?

In [ ]:
# Count how many of each type
type_counts = df['type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))

colors = ['#E50914', '#221F1F']  # Netflix red and dark grey

wedges, texts, autotexts = ax.pie(
    type_counts,
    labels=type_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=3)
)

for autotext in autotexts:
    autotext.set_fontsize(13)
    autotext.set_fontweight('bold')
    autotext.set_color('white')

ax.set_title('Netflix content: movies vs TV shows', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('chart_01_content_split.png')
plt.show()
print('Chart saved!')

### 3b. Content added over the years — when did Netflix grow?

In [ ]:
# Count titles added per year, split by type
yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)

# Only keep years with meaningful data (2010 onwards)
yearly = yearly[yearly.index >= 2010]

fig, ax = plt.subplots(figsize=(12, 6))

yearly.plot(kind='bar', ax=ax, color=['#E50914', '#564d4d'], width=0.7)

ax.set_title('Titles added to Netflix per year', fontsize=14, pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('Number of titles')
ax.legend(['Movie', 'TV Show'])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart_02_content_growth.png')
plt.show()
print('Chart saved!')

### 3c. Top 10 genres — what content dominates?

In [ ]:
# Genres are stored as comma-separated strings — we split them into individual rows
# e.g. 'Dramas, International Movies' becomes two separate entries
genres = df['listed_in'].str.split(', ').explode()
top_genres = genres.value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    top_genres.index[::-1],
    top_genres.values[::-1],
    color=sns.color_palette('Reds_r', 10)
)

for bar in bars:
    ax.text(
        bar.get_width() + 20,
        bar.get_y() + bar.get_height() / 2,
        f'{int(bar.get_width()):,}',
        va='center', fontsize=9
    )

ax.set_title('Top 10 genres on Netflix', fontsize=14, pad=15)
ax.set_xlabel('Number of titles')
plt.tight_layout()
plt.savefig('chart_03_top_genres.png')
plt.show()
print('Chart saved!')

### 3d. Top 10 countries producing Netflix content

In [ ]:
# Countries are also comma-separated — same split technique
countries = df['country'].str.split(', ').explode()
top_countries = countries[countries != 'Unknown'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    top_countries.index[::-1],
    top_countries.values[::-1],
    color=sns.color_palette('Blues_r', 10)
)

for bar in bars:
    ax.text(
        bar.get_width() + 10,
        bar.get_y() + bar.get_height() / 2,
        f'{int(bar.get_width()):,}',
        va='center', fontsize=9
    )

ax.set_title('Top 10 countries producing Netflix content', fontsize=14, pad=15)
ax.set_xlabel('Number of titles')
plt.tight_layout()
plt.savefig('chart_04_top_countries.png')
plt.show()
print('Chart saved!')

### 3e. What month does Netflix add the most content?

In [ ]:
# Map month numbers to names so the chart is readable
month_names = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

monthly = df['month_added'].value_counts().sort_index()
monthly.index = monthly.index.map(month_names)

fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(
    monthly.index,
    monthly.values,
    color=['#E50914' if v == monthly.max() else '#564d4d' for v in monthly.values]
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Which month does Netflix add the most content?', fontsize=14, pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Titles added')
plt.tight_layout()
plt.savefig('chart_05_monthly_additions.png')
plt.show()
print('Chart saved!')

### 3f. Content ratings distribution — who is Netflix making content for?

In [ ]:
# Filter out 'Unknown' and rare ratings
rating_counts = df[df['rating'] != 'Unknown']['rating'].value_counts()

# Keep only the main ratings
main_ratings = ['TV-MA', 'TV-14', 'TV-PG', 'R', 'PG-13', 'TV-Y7', 'TV-G', 'PG', 'TV-Y', 'G']
rating_counts = rating_counts[rating_counts.index.isin(main_ratings)]

fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(
    rating_counts.index,
    rating_counts.values,
    color=sns.color_palette('RdYlGn_r', len(rating_counts))
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 15,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Netflix content by audience rating', fontsize=14, pad=15)
ax.set_xlabel('Rating')
ax.set_ylabel('Number of titles')
plt.tight_layout()
plt.savefig('chart_06_ratings.png')
plt.show()
print('Chart saved!')

### 3g. Movie duration distribution — how long are Netflix movies?

In [ ]:
# Remove outliers — movies under 30 mins or over 250 mins are likely data errors
movies_clean = movies[(movies['duration_mins'] >= 30) & (movies['duration_mins'] <= 250)]

fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(
    movies_clean['duration_mins'],
    bins=30,
    color='#E50914',
    edgecolor='white',
    linewidth=0.5
)

# Add a vertical line showing the average
avg_duration = movies_clean['duration_mins'].mean()
ax.axvline(avg_duration, color='black', linestyle='--', linewidth=1.5,
           label=f'Average: {avg_duration:.0f} mins')

ax.set_title('Distribution of Netflix movie durations', fontsize=14, pad=15)
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Number of movies')
ax.legend()
plt.tight_layout()
plt.savefig('chart_07_movie_duration.png')
plt.show()
print(f'Average movie duration: {avg_duration:.0f} minutes')
print('Chart saved!')

---
## 4. Business recommendations
Based on the analysis above, here are concrete strategic recommendations.

In [ ]:
# Calculate the key numbers that support our recommendations
movie_pct = round(len(movies) / len(df) * 100, 1)
show_pct  = round(len(shows)  / len(df) * 100, 1)

top_genre      = genres.value_counts().index[0]
top_country    = countries[countries != 'Unknown'].value_counts().index[0]
peak_month_num = df['month_added'].value_counts().index[0]
peak_month     = month_names[peak_month_num]
top_rating     = df[df['rating'] != 'Unknown']['rating'].value_counts().index[0]

print('=' * 60)
print('  NETFLIX CONTENT STRATEGY — BUSINESS RECOMMENDATIONS')
print('=' * 60)

print(f'''
1. DOUBLE DOWN ON TV SHOWS
   Movies make up {movie_pct}% of content but TV shows ({show_pct}%)
   drive higher engagement through multi-episode viewing.
   Recommendation: shift investment ratio toward serialised content.

2. PRIORITISE THE TOP GENRE: {top_genre.upper()}
   This is the most represented genre in the catalogue.
   Recommendation: continue investing here while exploring
   adjacent genres to retain and expand the existing audience.

3. EXPAND INTERNATIONAL CONTENT BEYOND {top_country.upper()}
   {top_country} dominates production but global audiences
   are underserved. South Korean and Spanish content has
   already proven global appeal (Squid Game, Money Heist).
   Recommendation: increase co-productions with India,
   Japan, Brazil and South Korea.

4. FRONT-LOAD RELEASES IN {peak_month.upper()}
   {peak_month} is the peak month for new additions.
   Recommendation: align major original releases with
   this period to maximise subscriber acquisition.

5. MAINTAIN MATURE CONTENT FOCUS ({top_rating})
   {top_rating} is the dominant rating, confirming Netflix's
   core adult audience. Recommendation: preserve this focus
   while selectively expanding family content (TV-G, TV-Y)
   to reduce churn from households with children.
''')
print('=' * 60)

---
## 5. Summary statistics

In [ ]:
print('DATASET SUMMARY')
print(f'  Total titles analysed : {len(df):,}')
print(f'  Movies                : {len(movies):,} ({movie_pct}%)')
print(f'  TV Shows              : {len(shows):,} ({show_pct}%)')
print(f'  Countries represented : {df["country"].nunique():,}')
print(f'  Date range            : {df["year_added"].min():.0f} – {df["year_added"].max():.0f}')
print(f'  Avg movie duration    : {movies_clean["duration_mins"].mean():.0f} minutes')
print(f'  Missing values fixed  : director, cast, country, rating, duration')